In [1]:
import pandas as pd

In [2]:
# Load Parquet file phase 02
# from images
gov_effectiveness_df = pd.read_parquet("../data/image_data/wb_government_effectiveness.parquet")
edu_spending_df = pd.read_parquet("../data/image_data/Education_spending.parquet")
inflation_df = pd.read_parquet("../data/image_data/Inflation.parquet")
tax_revenue_df = pd.read_parquet("../data/image_data/Tax_revenue.parquet")
share_industry_df = pd.read_parquet("../data/image_data/Share_of_industry.parquet")
rule_of_law_df = pd.read_parquet("../data/image_data/wb_ruleoflaw.parquet")

In [47]:
df_list = [gov_effectiveness_df, edu_spending_df, inflation_df, tax_revenue_df, share_industry_df, rule_of_law_df]

In [4]:
name_df = ['gov_effectiveness_df', 'edu_spending_df', 'inflation_df', 'tax_revenue_df', 'share_industry_df', 'rule_of_law_df']

### Function: summarize_dataframes()
This function will:
- Accept a DataFrames with the name of the macroeconomic.
- Print for each:
- Shape
- Column types
- First few rows
- Summary stats

In [5]:
def summarize_dataframes(df, name, head_rows=3):
    """
    Summarizes a list of DataFrames with shape, dtypes, head, and describe.
    
    Parameters:
    - df: pandas DataFrames
    - names: name of the macroeconomico for the DataFrame
    - head_rows: number of rows to show in head() preview. By defaul the value is 3
    """
    print(f"\n{'='*40}\n📊 Summary for {name}\n{'='*40}")
    print(f"🔹 Shape: {df.shape}")
    print(f"🔹 Column Types:\n{df.dtypes}\n")
    print(f"🔹 First {head_rows} Rows:\n{df.head(head_rows)}\n")
    print(f"🔹 Descriptive Stats:\n{df.describe(include='all')}\n")

### count_columns_per_df(dfs, names=None)

Purpose:
Evaluate how many country columns each DataFrame has.

Output:
A formatted summary showing:
- DataFrame name
- Number of columns (i.e., countries)

Why:
Helps you understand which indicators have broader country coverage and which may need to be excluded or imputed.

In [6]:
def count_columns_per_df(dfs, names=None):
    """
    Returns the number of columns (countries) in each DataFrame.
    """
    if names is None:
        names = [f'DataFrame {i+1}' for i in range(len(dfs))]
    
    summary = []
    for name, df in zip(names, dfs):
        summary.append((name, df.shape[1]))
    
    print("\nColumn Count Summary")
    print(f"{'DataFrame':<25} {'# Columns':>10}")
    print("-" * 40)
    for name, count in summary:
        print(f"{name:<25} {count:>10}")

In [15]:
# phase 1 for comparision, do not run again
count_columns_per_df(df_list, name_df)


Column Count Summary
DataFrame                  # Columns
----------------------------------------
labor_force                       58
livestock_prod                    58
agricultural_ladn                 58
urban_pop                         58
gov_expenditure                   58
reserves                          60
investment_growth                 57
cpi                               60
gdp                               60
private_credit                    60
unemployment                      58
hpi                               59


In [7]:
count_columns_per_df(df_list, name_df)


Column Count Summary
DataFrame                  # Columns
----------------------------------------
gov_effectiveness_df              58
edu_spending_df                   58
inflation_df                      58
tax_revenue_df                    58
share_industry_df                 58
rule_of_law_df                    58


### summarize_time_structure_split(dfs, names=None)

Purpose:
Categorize each DataFrame based on how it stores time information:
- Either in the index (e.g., years like 2016, 2015)
- Or in explicit columns ('Year', 'Quarter')

Output:
Two formatted summaries:
- Time Structure Summary (Index-Based)
- DataFrame name
- Type of index
- Sample index value (e.g., 2016, 2015)
- Time Column Format Summary
- DataFrame name
- Type of 'Quarter' and 'Year' columns
- Sample values from those columns (e.g., 'Q1', 1960)

Why:
Helps you identify which DataFrames need time standardization before merging. Ensures consistent temporal alignment across datasets with mixed formats.

In [8]:
def summarize_time_structure_split(dfs, names=None):
    """
    Splits DataFrames into two groups:
    - Those with time stored in the index
    - Those with time stored in 'Year' and/or 'Quarter' columns

    Prints two formatted summaries.
    """
    if names is None:
        names = [f'DataFrame {i+1}' for i in range(len(dfs))]

    index_based = []
    column_based = []

    for name, df in zip(names, dfs):
        if 'Year' in df.columns or 'Quarter' in df.columns:
            quarter_type = type(df['Quarter'].iloc[0]).__name__ if 'Quarter' in df.columns else "N/A"
            year_type = type(df['Year'].iloc[0]).__name__ if 'Year' in df.columns else "N/A"
            sample_q = str(df['Quarter'].iloc[0]) if 'Quarter' in df.columns else "N/A"
            sample_y = str(df['Year'].iloc[0]) if 'Year' in df.columns else "N/A"
            column_based.append((name, quarter_type, year_type, sample_q, sample_y))
        else:
            index_type = type(df.index[0]).__name__
            sample_time = str(df.index[0])
            index_based.append((name, index_type, sample_time))

    # Print index-based summary
    print("\n📅 Time Structure Summary (Index-Based)")
    print(f"{'DataFrame':<25} {'Index Type':<15} {'Sample Time'}")
    print("-" * 55)
    for name, idx_type, sample in index_based:
        print(f"{name:<25} {idx_type:<15} {sample}")

    # Print column-based summary
    print("\n🕰️ Time Column Format Summary")
    print(f"{'DataFrame':<25} {'Quarter Type':<15} {'Year Type':<15} {'Sample Quarter':<15} {'Sample Year'}")
    print("-" * 80)
    for name, qt, yt, sq, sy in column_based:
        print(f"{name:<25} {qt:<15} {yt:<15} {sq:<15} {sy}")

In [9]:
summarize_time_structure_split(df_list, name_df)


📅 Time Structure Summary (Index-Based)
DataFrame                 Index Type      Sample Time
-------------------------------------------------------
gov_effectiveness_df      int64           2016
edu_spending_df           int64           2010
inflation_df              int64           2017
tax_revenue_df            int64           2015
share_industry_df         int64           2016
rule_of_law_df            int64           2016

🕰️ Time Column Format Summary
DataFrame                 Quarter Type    Year Type       Sample Quarter  Sample Year
--------------------------------------------------------------------------------


In [10]:
def sample_indices_named(df_list, df_names):
    """
    Takes a list of DataFrames and their names, returns a dictionary
    with one sampled index per DataFrame as native Python int.
    """
    samples = {}
    for df, name in zip(df_list, df_names):
        if not df.empty:
            sampled_index = df.index.to_series().sample(1).iloc[0]
            samples[name] = int(sampled_index)  # Convert to native int
        else:
            samples[name] = None
    return samples

In [11]:
sample_indices_named(df_list,name_df)

{'gov_effectiveness_df': 2020,
 'edu_spending_df': 2012,
 'inflation_df': 2019,
 'tax_revenue_df': 2021,
 'share_industry_df': 2018,
 'rule_of_law_df': 2017}

### count_missing_series(dfs, names=None)

Purpose:
Count how many country columns in each DataFrame contain missing values.

Output:
A summary showing:
- DataFrame name
- Number of columns with missing values
- Total number of columns

Why:
Helps you assess data quality and decide whether to drop, impute, or prioritize certain indicators.

In [12]:
def count_missing_series(dfs, names=None):
    """
    Counts how many columns (countries) have missing values in each DataFrame.
    """
    if names is None:
        names = [f'DataFrame {i+1}' for i in range(len(dfs))]
    
    summary = []
    for name, df in zip(names, dfs):
        missing_cols = df.isnull().any().sum()
        total_cols = df.shape[1]
        summary.append((name, missing_cols, total_cols))
    
    print("\n❓ Missing Series Summary")
    print(f"{'DataFrame':<25} {'Missing':>10} {'Total':>10}")
    print("-" * 45)
    for name, missing, total in summary:
        print(f"{name:<25} {missing:>10} {total:>10}")

In [13]:
count_missing_series(df_list, name_df)


❓ Missing Series Summary
DataFrame                    Missing      Total
---------------------------------------------
gov_effectiveness_df               0         58
edu_spending_df                   58         58
inflation_df                       3         58
tax_revenue_df                     6         58
share_industry_df                 58         58
rule_of_law_df                     1         58


### check_time_alignment(dfs, names=None)

Purpose:
Evaluate whether all DataFrames share a consistent time structure—either by year, quarter, or a unified format. This is critical before merging datasets for correlation or similarity analysis.

Why It Matters:
- Ensures that indicators are temporally aligned
- Prevents mismatches when merging
- Helps decide whether to aggregate quarterly data to annual


In [14]:
def check_time_alignment(dfs, names=None):
    """
    Checks the time alignment of each DataFrame by inspecting its time-related columns or index.
    Returns the unique time keys (e.g., years or quarters) for each DataFrame.
    """
    if names is None:
        names = [f'DataFrame {i+1}' for i in range(len(dfs))]

    print("\n🧭 Time Alignment Diagnostic")
    print(f"{'DataFrame':<25} {'# Unique Time Points':>20} {'Sample Time Keys'}")
    print("-" * 70)

    for name, df in zip(names, dfs):
        if 'Year' in df.columns:
            time_keys = df['Year'].astype(str)
            if 'Quarter' in df.columns:
                time_keys = time_keys + "-Q" + df['Quarter'].astype(str)
        else:
            time_keys = df.index.astype(str)

        unique_keys = sorted(time_keys.unique())
        print(f"{name:<25} {len(unique_keys):>20} {unique_keys[:3]}{'...' if len(unique_keys) > 3 else ''}")

In [15]:
check_time_alignment(df_list, name_df)


🧭 Time Alignment Diagnostic
DataFrame                 # Unique Time Points Sample Time Keys
----------------------------------------------------------------------
gov_effectiveness_df                         8 ['2016', '2017', '2018']...
edu_spending_df                             14 ['2010', '2011', '2012']...
inflation_df                                 8 ['2017', '2018', '2019']...
tax_revenue_df                               8 ['2015', '2016', '2017']...
share_industry_df                            9 ['2016', '2017', '2018']...
rule_of_law_df                               8 ['2016', '2017', '2018']...


Once run it, we’ll know which datasets are aligned and which need transformation.

### check_indicator_variance(dfs, names=None)

Purpose:
Evaluate whether each macroeconomic indicator shows sufficient variability across countries and time. Indicators with low variance may not contribute meaningful signal to correlation or similarity analysis.

Why It Matters:
- Low variance indicators are often uninformative—they don’t differentiate countries or explain variation in HPI.
- High variance may indicate strong signal, but also potential outliers or scale issues.
- Helps you decide which indicators to prioritize, transform, or exclude.

In [16]:
def check_indicator_variance(dfs, names=None):
    """
    Computes the average standard deviation across country columns for each DataFrame.
    Assumes countries are columns and time is in index or separate columns.
    """
    if names is None:
        names = [f'DataFrame {i+1}' for i in range(len(dfs))]

    print("\n📈 Indicator Variance Diagnostic")
    print(f"{'DataFrame':<25} {'Avg Std Dev':>15} {'Max Std Dev':>15} {'Min Std Dev':>15}")
    print("-" * 70)

    for name, df in zip(names, dfs):
        country_cols = [col for col in df.columns if col not in ['Year', 'Quarter']]
        stds = df[country_cols].std(skipna=True)
        avg_std = stds.mean()
        max_std = stds.max()
        min_std = stds.min()
        print(f"{name:<25} {avg_std:>15.2f} {max_std:>15.2f} {min_std:>15.2f}")

In [17]:
check_indicator_variance(df_list, name_df)


📈 Indicator Variance Diagnostic
DataFrame                     Avg Std Dev     Max Std Dev     Min Std Dev
----------------------------------------------------------------------
gov_effectiveness_df                 0.11            0.27            0.04
edu_spending_df                      0.34            1.39            0.09
inflation_df                         3.24           24.80            0.76
tax_revenue_df                       0.91            5.38            0.27
share_industry_df                    1.08            7.03            0.30
rule_of_law_df                       0.08            0.18            0.02


| Column | Meaning | Why It Matters |
|--------|---------|----------------|
| **Avg Std Dev** | Average standard deviation across all country columns | Gives a sense of overall variability in the indicator |
| **Max Std Dev** | Highest standard deviation among countries | Flags countries with extreme fluctuations or outliers |
| **Min Std Dev** | Lowest standard deviation among countries | Reveals flat or uninformative series |

---

**High Variance Indicators**
These have strong signal potential but may need normalization due to scale:

These are **informative but scale-sensitive**. 

---

**Moderate Variance Indicators**
These are more stable and interpretable:

These are **well-behaved**

---

**Low Variance Indicators**
These may be too flat to add value:

These are **candidates for exclusion or transformation** unless they show strong correlation with HPI.

---

**What to Do Next**

- **Normalize** high-variance indicators before similarity modeling.Even indicators with moderate variance (like investment growth or unemployment) should be normalized if they’re going into a similarity or clustering model. Not because they’re problematic, but because you want a level playing field across all features.
- **Transform or exclude** low-variance indicators unless they correlate well with HPI.
- **Use this diagnostic** alongside your correlation matrix to select a diverse, non-redundant feature set.

In [20]:
import os

In [18]:
def verificar_imagenes_faltantes(directorio_madre, indicadores):
    faltantes = []

    # Recorremos cada carpeta (país) dentro del directorio madre
    for nombre_pais in os.listdir(directorio_madre):
        ruta_pais = os.path.join(directorio_madre, nombre_pais)
        if not os.path.isdir(ruta_pais):
            continue  # Ignorar si no es carpeta

        archivos_png = [f for f in os.listdir(ruta_pais) if f.endswith('.png')]

        # Verificamos cada indicador
        for indicador in indicadores:
            esperado = f"{indicador}{nombre_pais}.png"
            if not any(esperado in archivo for archivo in archivos_png):
                faltantes.append((nombre_pais, esperado))

    return faltantes

In [21]:
indicadores = [
    "Education_spending_",
    "Inflation_",
    "Share_of_industry_",
    "Tax_revenue_",
    "wb_government_effectiveness_",
    "wb_ruleoflaw_"
]
faltantes = verificar_imagenes_faltantes("../plots_2/", indicadores)

for pais, archivo in faltantes:
    print(f"Falta en {pais}: {archivo}")

Falta en Canada: Share_of_industry_Canada.png
Falta en Hong-Kong: Tax_revenue_Hong-Kong.png
Falta en India: Tax_revenue_India.png
Falta en Indonesia: Inflation_Indonesia.png
Falta en Macedonia: Education_spending_Macedonia.png
Falta en Macedonia: Tax_revenue_Macedonia.png
Falta en New-Zealand: Share_of_industry_New-Zealand.png
Falta en New-Zealand: Tax_revenue_New-Zealand.png
Falta en Peru: Tax_revenue_Peru.png
Falta en Russia: Inflation_Russia.png


#### fuentes de datos faltantes

* canada: https://www.theglobaleconomy.com/Canada/Share_of_industry/

* education colombia: https://tradingeconomics.com/colombia/public-spending-on-education-total-percent-of-gdp-wb-data.html

* hong kong es china y china ya tenemos

* india: https://incometaxindia.gov.in/Documents/Direct%20Tax%20Data/Final-Approved-Time-Series-Data-2023-24-English.pdf

* indonesia no tenemos entre los paises
* macedonia no tenemos entre los paises

* new-zealand: https://www.theglobaleconomy.com/New-Zealand/Share_of_industry/

* new-zealand: https://tradingeconomics.com/new-zealand/tax-revenue-percent-of-gdp-wb-data.html

* peru:

https://www.ceicdata.com/en/indicator/peru/tax-revenue
https://tradingeconomics.com/peru/tax-revenue-percent-of-gdp-wb-data.html
https://www.oecd.org/content/dam/oecd/en/publications/reports/2025/05/revenue-statistics-in-latin-america-and-the-caribbean-2025-country-notes_29961c77/peru_4b1aefef/f7ed6d9a-en.pdf
https://www.statista.com/statistics/1390855/tax-revenue-as-a-share-of-gdp-peru/

* russia:

https://www.statista.com/statistics/271376/inflation-rate-in-russia/
https://www.macrotrends.net/global-metrics/countries/rus/russia/inflation-rate-cpi
https://www.theglobaleconomy.com/Russia/Inflation/

## Canada

### Share of industry

In [23]:
share_industry_df.tail()

,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
2020,25.39,25.65,19.08,19.51,21.82,NaN,30.86,36.91,23.44,20.10,...,28.94,23.34,32.54,20.12,21.81,24.83,33.29,28.03,17.12,17.41
2021,25.42,25.73,18.00,22.12,20.74,NaN,31.03,38.09,24.56,19.58,...,28.17,24.87,32.43,20.32,23.06,25.64,35.03,31.13,17.50,16.73
2022,27.48,25.21,19.51,22.82,25.30,NaN,28.80,37.89,26.12,20.07,...,28.01,24.93,31.73,20.93,24.02,24.54,35.12,31.29,18.08,17.24
2023,27.65,25.24,18.54,22.14,22.66,NaN,30.09,36.77,24.69,21.01,...,29.70,24.62,31.59,20.12,23.23,24.67,32.98,28.38,17.75,17.49
2024,25.97,23.06,17.58,21.33,22.49,NaN,NaN,36.48,23.07,19.84,...,28.76,24.41,NaN,19.50,22.57,24.72,32.13,25.94,17.33,16.74


In [24]:
print(type(share_industry_df.index))


<class 'pandas.core.indexes.base.Index'>


In [25]:
print(share_industry_df.index.dtype)

int64


In [26]:
canada_share_industry = {
    2014: 27.14,
    2015: 24.42,
    2016: 23.30,
    2017: 24.03,
    2018: 24.62,
    2019: 24.08,
    2020: 22.45,
    2021: 25.33
}

share_industry_df['Canada'] = canada_share_industry
share_industry_df.tail()

,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
2020,25.39,25.65,19.08,19.51,21.82,22.45,30.86,36.91,23.44,20.10,...,28.94,23.34,32.54,20.12,21.81,24.83,33.29,28.03,17.12,17.41
2021,25.42,25.73,18.00,22.12,20.74,25.33,31.03,38.09,24.56,19.58,...,28.17,24.87,32.43,20.32,23.06,25.64,35.03,31.13,17.50,16.73
2022,27.48,25.21,19.51,22.82,25.30,NaN,28.80,37.89,26.12,20.07,...,28.01,24.93,31.73,20.93,24.02,24.54,35.12,31.29,18.08,17.24
2023,27.65,25.24,18.54,22.14,22.66,NaN,30.09,36.77,24.69,21.01,...,29.70,24.62,31.59,20.12,23.23,24.67,32.98,28.38,17.75,17.49
2024,25.97,23.06,17.58,21.33,22.49,NaN,NaN,36.48,23.07,19.84,...,28.76,24.41,NaN,19.50,22.57,24.72,32.13,25.94,17.33,16.74


In [27]:
share_industry_df.head()

,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
2016,NaN,NaN,NaN,NaN,NaN,23.30,NaN,NaN,NaN,NaN,...,NaN,NaN,34.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017,23.47,25.80,19.15,18.19,23.99,24.03,27.86,39.13,26.81,19.77,...,28.93,23.61,34.77,19.90,22.16,24.22,35.02,29.13,18.00,17.81
2018,24.12,25.72,18.96,18.75,22.19,24.62,27.12,38.98,26.90,19.29,...,28.07,23.54,34.05,19.74,22.19,24.42,34.74,29.44,18.20,17.90
2019,25.27,25.54,19.24,18.75,21.52,24.08,29.83,37.76,25.99,19.58,...,28.56,23.62,32.68,19.98,22.29,24.72,33.59,27.33,17.86,17.76
2020,25.39,25.65,19.08,19.51,21.82,22.45,30.86,36.91,23.44,20.10,...,28.94,23.34,32.54,20.12,21.81,24.83,33.29,28.03,17.12,17.41


## New-Zealand

### Share of industry

In [29]:
share_industry_df[['New-Zealand']].tail()

,New-Zealand
2020,NaN
2021,NaN
2022,NaN
2023,NaN
2024,NaN


In [31]:
new_zealand_share_industry = {
    2015: 21.21,
    2016: 20.17,
    2017: 20.53,
    2018: 20.38,
    2019: 20.37,
    2020: 19.98,
    2021: 18.83,
    2022: 19.60
}

share_industry_df['New-Zealand'] = new_zealand_share_industry
share_industry_df[['New-Zealand']].tail()

,New-Zealand
2020,19.98
2021,18.83
2022,19.60
2023,NaN
2024,NaN


### Tax Revenue

In [32]:
tax_revenue_df[['New-Zealand']].tail()

,New-Zealand
2018,NaN
2019,NaN
2020,NaN
2021,NaN
2022,NaN


In [33]:
new_zealand_tax_revenue = {
    2015: 27.39,
    2016: 27.25,
    2017: 27.31,
    2018: 27.46,
    2019: 28.15,
    2020: 27.33,
    2021: 29.02,
    2022: 29.60
}

tax_revenue_df['New-Zealand'] = new_zealand_tax_revenue
tax_revenue_df[['New-Zealand']].tail()

,New-Zealand
2018,27.46
2019,28.15
2020,27.33
2021,29.02
2022,29.60


## India

### Tax revenue

In [34]:
tax_revenue_df[['India']].tail()

,India
2018,NaN
2019,NaN
2020,NaN
2021,NaN
2022,NaN


In [35]:
india_tax_revenue = {
    2016: 5.53,
    2017: 5.86,
    2018: 6.02,
    2019: 5.23,
    2020: 4.78,
    2021: 5.97,
    2022: 6.17,
    2023: 6.64
}

new_df = pd.DataFrame({'India': india_tax_revenue})


tax_revenue_df = tax_revenue_df.combine_first(new_df)
tax_revenue_df[['India']].tail()

,India
2019,5.23
2020,4.78
2021,5.97
2022,6.17
2023,6.64


In [38]:
tax_revenue_df[['India']].head()

,India
2015,NaN
2016,5.53
2017,5.86
2018,6.02
2019,5.23


## Peru

### Tax revenue

In [39]:
tax_revenue_df[['Peru']].tail()

,Peru
2019,NaN
2020,NaN
2021,NaN
2022,NaN
2023,NaN


In [40]:
peru_tax_revenue = {
    2014: 16.72,
    2015: 14.91,
    2016: 13.78,
    2017: 13.17,
    2018: 14.28,
    2019: 14.52,
    2020: 13.22,
    2021: 15.92
}

tax_revenue_df['Peru'] = peru_tax_revenue
tax_revenue_df[['Peru']].tail()

,Peru
2019,14.52
2020,13.22
2021,15.92
2022,NaN
2023,NaN


## Russia

### Inflation

In [41]:
inflation_df[['Russia']].tail()

,Russia
2020,NaN
2021,NaN
2022,NaN
2023,NaN
2024,NaN


In [43]:
russia_inflation = {
    2014: 7.80,
    2015: 15.50,
    2016: 7.00,
    2017: 3.70,
    2018: 2.90,
    2019: 4.50,
    2020: 3.40,
    2021: 6.70
}

inflation_df['Russia'] = russia_inflation
inflation_df[['Russia']].tail()

,Russia
2020,3.4
2021,6.7
2022,NaN
2023,NaN
2024,NaN


## Colombia
### Education spending

In [44]:
edu_spending_df[['Colombia']].tail()

,Colombia
2019,NaN
2020,NaN
2021,NaN
2022,NaN
2023,NaN


In [45]:
colombia_education_spending = {
    2016: 4.4,
    2017: 4.5,
    2018: 4.4,
    2019: 4.5,
    2020: 5.1,
    2021: 4.7,
    2022: 4.2,
    2023: 4.3
}

edu_spending_df['Colombia'] = colombia_education_spending
edu_spending_df[['Colombia']].tail()

,Colombia
2019,4.5
2020,5.1
2021,4.7
2022,4.2
2023,4.3


In [48]:
count_missing_series(df_list, name_df)


❓ Missing Series Summary
DataFrame                    Missing      Total
---------------------------------------------
gov_effectiveness_df               0         58
edu_spending_df                   58         58
inflation_df                       3         58
tax_revenue_df                    58         58
share_industry_df                 58         58
rule_of_law_df                     1         58


In [52]:
inflation_nan = inflation_df.columns[inflation_df.isna().any()]
print(inflation_nan)

Index(['Bulgaria', 'Indonesia', 'Russia'], dtype='object')


In [54]:
inflation_df[['Bulgaria','Indonesia','Russia']]

,Bulgaria,Indonesia,Russia
2017,2.8,NaN,3.7
2018,3.1,NaN,2.9
2019,1.7,NaN,4.5
2020,3.3,NaN,3.4
2021,15.3,NaN,6.7
2022,9.4,NaN,NaN
2023,2.4,NaN,NaN
2024,NaN,NaN,NaN


- we do not need indonesia

In [53]:
rule_of_law_nan = rule_of_law_df.columns[rule_of_law_df.isna().any()]
print(rule_of_law_nan)

Index(['Switzerland'], dtype='object')


In [55]:
rule_of_law_df[['Switzerland']]

,Switzerland
2016,NaN
2017,NaN
2018,NaN
2019,NaN
2020,NaN
2021,NaN
2022,NaN
2023,NaN


## Switzerland
### Rule of law

In [56]:
switzerland_rule_law = {
    2016: 1.91,
    2017: 1.89,
    2018: 1.89,
    2019: 1.86,
    2020: 1.78,
    2021: 1.77,
    2022: 1.75,
    2023: 1.76
}

rule_of_law_df['Switzerland'] = switzerland_rule_law
rule_of_law_df[['Switzerland']].tail()

,Switzerland
2019,1.86
2020,1.78
2021,1.77
2022,1.75
2023,1.76


In [61]:
all_nan_columns = edu_spending_df.columns[edu_spending_df.isna().all()]
print(all_nan_columns)

Index(['Macedonia'], dtype='object')


- we do not need macedonia

In [63]:
all_nan_columns = tax_revenue_df.columns[tax_revenue_df.isna().all()]
print(all_nan_columns)

Index(['Hong-Kong', 'Macedonia'], dtype='object')


- china is the same that hong kong and we do have china statistics
- we do not need macedonia

In [64]:
all_nan_columns = share_industry_df.columns[share_industry_df.isna().all()]
print(all_nan_columns)

Index([], dtype='object')


In [65]:
gov_effectiveness_df.to_parquet("../data/image_data/wb_government_effectiveness.parquet")
edu_spending_df.to_parquet("../data/image_data/Education_spending.parquet")
inflation_df.to_parquet("../data/image_data/Inflation.parquet")
tax_revenue_df.to_parquet("../data/image_data/Tax_revenue.parquet")
share_industry_df.to_parquet("../data/image_data/Share_of_industry.parquet")
rule_of_law_df.to_parquet("../data/image_data/wb_ruleoflaw.parquet")